In [ ]:
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])
    
if 'date' in df.columns:
    df = df.sort_values('date').reset_index(drop=True)



In [ ]:
def clean_csv(filename):
    
    filepath = filename
    print(f"\n--- Cleaning {filename} ---")
 
    # 1. Load the data
    df = pd.read_csv(filepath, names=['id', 'date', 'user', 'pc', 'activity'], header=0)
    print(f"Original shape: {df.shape}")
 
    
    missing_counts = df.isnull().sum()
    print("Missing values per column:")
    print(missing_counts[missing_counts > 0])
 
    
    df = df.dropna()
    print(f"Shape after dropping missing values: {df.shape}")
 
    
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    print(f"Removed {before - after} duplicate rows")
 
    
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        
        before = df.shape[0]
        df = df.dropna(subset=['date'])
        after = df.shape[0]
        if before != after:
            print(f"Dropped {before - after} rows with bad timestamps")
 
    
    if 'date' in df.columns:
        df = df.sort_values('date').reset_index(drop=True)
 
    print(f"Final shape: {df.shape}")
    print(df.columns.tolist())
 
 
    return df
 

files_to_clean = ["/kaggle/input/datasets/ananyassssss/rvudata/device.csv", "/kaggle/input/datasets/ananyassssss/rvudata/http.csv", "/kaggle/input/datasets/ananyassssss/rvudata/logon.csv"]
 
cleaned_dataframes = {}
for f in files_to_clean:
    cleaned_dataframes[f] = clean_csv(f)

print(cleaned_dataframes)
 
    

In [ ]:
# df3=pd.merge(cleaned_dataframes[0],cleaned_dataframes[1],cleaned_dataframes[2],on='id')
# df3

logon_df = cleaned_dataframes["/kaggle/input/datasets/ananyassssss/rvudata/logon.csv"].copy()
device_df = cleaned_dataframes["/kaggle/input/datasets/ananyassssss/rvudata/device.csv"].copy()
http_df = cleaned_dataframes["/kaggle/input/datasets/ananyassssss/rvudata/http.csv"].copy()

logon_df['source'] = 'logon'
device_df['source'] = 'device'
http_df['source'] = 'http'

combined = pd.concat([logon_df, device_df, http_df], ignore_index=True)
combined = combined.sort_values(['user', 'date']).reset_index(drop=True)

combined.head(5)

In [ ]:
def extract_daily_indicators(combined):
    
    combined['day'] = combined['date'].dt.date
 
    # 2. LOGIN FREQUENCY: count logon events per user per day
    logon_only = combined[combined['source'] == 'logon']
    login_freq = (
        logon_only.groupby(['user', 'day'])
        .size()
        .reset_index(name='login_count')
    )
 
    # 3. DEVICE USAGE: count device connect/disconnect events per user per day
    device_only = combined[combined['source'] == 'device']
    device_usage = (
        device_only.groupby(['user', 'day'])
        .size()
        .reset_index(name='device_event_count')
    )
    # 4. HTTP / WEB ACTIVITY VOLUME: count http events per user per day
    http_only = combined[combined['source'] == 'http']
    http_volume = (
        http_only.groupby(['user', 'day'])
        .size()
        .reset_index(name='http_event_count')
    )
 
    # 5. UNIQUE PCs USED per user per day (across all activity)
    unique_pcs = (
        combined.groupby(['user', 'day'])['pc']
        .nunique()
        .reset_index(name='unique_pc_count')
    )
 
    
    daily_indicators = unique_pcs.merge(
        login_freq, on=['user', 'day'], how='left'
    ).merge(
        device_usage, on=['user', 'day'], how='left'
    ).merge(
        http_volume, on=['user', 'day'], how='left'
    )
 
    
    count_cols = ['login_count', 'device_event_count', 'http_event_count']
    daily_indicators[count_cols] = daily_indicators[count_cols].fillna(0).astype(int)
 
    return daily_indicators


daily_indicators = extract_daily_indicators(combined)
 
print(f"Daily indicators shape: {daily_indicators.shape}")
print(daily_indicators.head(15))
 
   

In [ ]:
def build_baseline_profiles(daily_indicators):
    indicator_cols = ['login_count', 'device_event_count',
                       'http_event_count', 'unique_pc_count']
 
    #Group by user and calculate mean + std for each indicator
    
    baseline = (
        daily_indicators.groupby('user')[indicator_cols]
        .agg(['mean', 'std'])
    )
 
   
    baseline.columns = [f"{col}_{stat}" for col, stat in baseline.columns]
    baseline = baseline.reset_index()
 
   
    baseline = baseline.fillna(0)
 
    return baseline

baseline_profiles = build_baseline_profiles(daily_indicators)
 
print(f"Number of user profiles: {baseline_profiles.shape[0]}")
print(baseline_profiles.head(10))